In [ ]:

!pip install langchain_classic langchain-core langchain-community langchain-chroma langchain-huggingface sentence-transformers chromadb optuna unstructured


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Colab Notebooks/side_projects/RAG_project"

import sys
sys.path.append(base_path)

from datasets import training_set, calibration_set, test_set
from rag_functions import run_trial, BO_pipeline

import numpy as np
import optuna
import math
import optuna.visualization as vis

from transformers import pipeline
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_classic.schema import Document

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA

import optuna
import numpy as np
from optuna.samplers import TPESampler



In [ ]:


with open(f"{base_path}data/template.txt", "r") as f:
    template = f.read()

with open(f"{base_path}data/omniatech.txt", "r") as f:
    omniatech = f.read()

omniatech_to_chunk = omniatech#[: 10*1000]



embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

fixing_param_dict = {}

max_tokens_list = [64, 128, 256, 512, 1024]
penalty_list = [1.0, 1.25, 1.5, 1.75, 2.0]
temperature_list = [0.05, 0.1, 0.15, 0.2, 0.3]
top_k_list = [1, 2, 3, 4]
chunk_size_list = [100, 200, 300, 400]
chunk_overlap_list = [0.1, 0.2, 0.3, 0.4, 0.5]

param_list_dict = {
    "max_tokens": max_tokens_list,
    "penalty": penalty_list,
    "temperature": temperature_list,
    "top_k": top_k_list,
    "chunk_size": chunk_size_list,
    "chunk_overlap": chunk_overlap_list
}
param_name_list = list(param_list_dict.keys())

QA_CHAIN_PROMPT = ChatPromptTemplate.from_messages([
            ("system", template),
            ("human", "{question}")
        ])

chosen_param = {}
fixing_param_dict = {}
all_param_fixed = False

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

chosen_param_list = [] # to analyse what changes across iterations
while True:
    bay_conf = 20
    sampler_seed = 42
    sampler = TPESampler(seed = sampler_seed)
    study = optuna.create_study(direction = "maximize", sampler = sampler)

    study, scores = BO_pipeline(bay_conf, study, param_list_dict, fixing_param_dict, QA_CHAIN_PROMPT, calibration_set, omniatech_to_chunk, embedding_model, base_model, tokenizer)



    importances = optuna.importance.get_param_importances(study) # check the hyperparametes weigths on to choose  those to fix

    for param, importance in importances.items():
        print(f"{param}: {importance:.4f}")

    # Select hyper that have importance lower than 15% (might be changed)
    best_parameters = study.best_params
    chosen_param.update(best_parameters)
    fixing_parameters = [name for name, value in importances.items() if value < 0.15]


    for element in fixing_parameters:
      fixing_param_dict[element] = best_parameters[element]

    chosen_param_list.append(chosen_param.copy())

    if len(fixing_parameters) == 0 or len(fixing_param_dict) >= len(param_name_list) - 1:
        break


print('chosen_param', chosen_param)
q_hat = run_trial(chosen_param, omniatech_to_chunk, embedding_model, QA_CHAIN_PROMPT, calibration_set, base_model, tokenizer, phase = 'calibration')
ncs_scores = run_trial(chosen_param, omniatech_to_chunk, embedding_model, QA_CHAIN_PROMPT, test_set, base_model, tokenizer, phase = 'test', q_hat = q_hat)